In [69]:
# 

# Demo: Adaptive Staircase with N-AFC

This notebook runs an adaptive **transformed up/down staircase** on top of the
`whispy.ui.NAFC` window. The staircase picks soundfiles of increasing intensity
and converges on the participant's threshold using, e.g., a 1-up/2-down rule.

If you haven't installed the project yet, run this from the repo root:
```bash
pip install -e .
```

The notebook has two ways to run the staircase:
- **Section 4** opens the real N-AFC windows (interactive ‒ you click through trials).
- **Section 6** simulates an observer so the whole notebook runs without a GUI.

In [70]:
%reload_ext autoreload

%autoreload 2

import whispy
from pathlib import Path
import pandas as pd
from whispy.interfaces import SoundDevice
from whispy.ui.n_afc import NAFC
import importlib as imp

imp.reload(whispy.ui.n_afc)


<module 'whispy.ui.n_afc' from 'D:\\Studium\\TUB\\SoSe_26\\Python_und_Akustik\\Whispy\\whispy\\whispy\\ui\\n_afc.py'>

## 1. Stimuli & handler

We reuse the example stimuli (`configs/stimuli.yml`, ids `1..4`) and the
`SoundDevice` handler. For this demo we treat the stimulus ids as levels of
increasing intensity. In a real experiment, order `levels` by discriminability
(e.g. soundfiles of increasing signal level).

In [71]:
config_dir = Path('..') / 'configs'
stimuli_dir = Path('..') / 'stimuli'

stimuli_handler = SoundDevice(config_dir / 'stimuli.yml', stimuli_dir, loop=False)
print('Loaded stimuli:', list(stimuli_handler.stimuli.keys()))

Loaded stimuli: [1, 2, 3, 4]


## 2. Map an intensity level to an N-AFC trial

The staircase only tracks *which* level to present; you decide what each trial
looks like via a `build_screen(level)` callback. Here we build a 3-AFC
\"odd one out\" trial: two intervals play a fixed **standard** (stimulus id 1)
and one interval plays the current **target** (`level`). A higher level is
easier to tell apart from the standard.

In [72]:
standard = 1
levels = [2, 3, 4]  # stimulus ids, ordered by increasing intensity / discriminability

def build_screen(level):
    return {
        'block': 0,
        'section': 0,
        'attribute': 'difference',
        'test': [standard, standard, level],  # NAFC shuffles button order
        'correct': level,
        'block_name': 'Staircase',
        'section_name': '3-AFC',
    }

build_screen(levels[0])

{'block': 0,
 'section': 0,
 'attribute': 'difference',
 'test': [1, 1, 2],
 'correct': 2,
 'block_name': 'Staircase',
 'section_name': '3-AFC'}

## 3. Create the staircase

`n_up=1` / `n_down=2` means: step **up** (easier) after one wrong answer, step
**down** (harder) after two correct in a row. The run stops at whichever comes
first: `max_reversals` reversals or `max_trials` trials.

In [73]:
staircase = whispy.Staircase(
    levels,
    build_screen=build_screen,
    n_up=1,                          # step up (easier) after 1 wrong
    n_down=2,                        # step down (harder) after 2 correct
    step=1,
    start_index=len(levels) - 1,     # start at the easiest level
    max_reversals=6,
    max_trials=1,
)
print('start level:', staircase.current_level, '| finished:', staircase.finished)

start level: 4 | finished: False


## 4. Run with the real N-AFC window (interactive)

Running the next cell opens **one N-AFC window per trial** until the staircase
stops. In each window: click a button to hear that interval, then press
**\"Submit choice\"**. `staircase.run(...)` drives the whole loop and returns the
trial history.

In [74]:
def run_trial(screen):
    naf = NAFC(screen=screen, stimuli_handler=stimuli_handler, blocking=True, debug=True)
    return bool(naf.get_results()['correct_bool'].iloc[0])

results = staircase.run(run_trial)
results

,trial,level_index,level,correct,step,reversal
0,1,2,4,False,up,False


## 5. Inspect results & threshold

The threshold is the mean level over the final reversals (in level units when
the levels are numeric).

In [75]:
print('trials        :', len(results))
print('reversals     :', staircase.n_reversals)
print('reversal levels:', staircase.reversal_levels())
print('threshold     :', staircase.threshold())
results

trials        : 1
reversals     : 0
reversal levels: []
threshold     : None


,trial,level_index,level,correct,step,reversal
0,1,2,4,False,up,False


## 6. (Optional) Simulate responses without a GUI

To run the whole notebook end-to-end without clicking, replace the real window
with a simulated observer whose accuracy increases with level (chance is 1/3 for
a 3-AFC). This reuses the same `build_screen` and staircase configuration.

In [76]:
import numpy as np

rng = np.random.default_rng(0)

# Probability of a correct response at each target level (chance = 1/3).
p_correct = {2: 0.5, 3: 0.75, 4: 0.95}

def simulated_trial(screen):
    level = screen['correct']
    return bool(rng.random() < p_correct[level])

sim = whispy.Staircase(
    levels, build_screen=build_screen,
    n_up=1, n_down=2, start_index=len(levels) - 1,
    max_reversals=6, max_trials=30,
)
sim_results = sim.run(simulated_trial)
print('simulated threshold:', sim.threshold())
sim_results

simulated threshold: 3.3333333333333335


,trial,level_index,level,correct,step,reversal
0,1,2,4,True,,False
1,2,2,4,True,down,False
2,3,1,3,True,,False
3,4,1,3,True,down,False
4,5,0,2,False,up,True
5,6,1,3,False,up,False
6,7,2,4,True,,False
7,8,2,4,True,down,True
8,9,1,3,True,,False
9,10,1,3,False,up,True
